# 🤖 Research Paper AI Assistant — Enhanced Accuracy Edition

## 📌 What's improved over the basic version

| Problem in basic version | Fix in this version |
|---|---|
| Only 3 small chunks retrieved | Retrieves **6 chunks** with larger size (1000 chars) |
| Model had no overview of the paper | **Full-document summary** built at upload time and injected into every prompt |
| Vague prompts | **Specific, structured prompts** tailored per task (Q&A / summarize / keypoints) |
| Chunks cut mid-sentence | Chunks now respect **sentence boundaries** with 200-char overlap |
| No answer quality control | **Post-processing** cleans and validates model output |
| Single retrieval pass | **Two-stage retrieval**: broad search + keyword re-rank |

**Model stays the same:** `google/flan-t5-base` — no extra download size.

---

## 🌐 API Endpoints (same as before — fully compatible with your MERN app)

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET`  | `http://localhost:8000/health` | Check if API is running |
| `POST` | `http://localhost:8000/upload` | Upload a PDF file |
| `POST` | `http://localhost:8000/chat` | Ask a question |
| `POST` | `http://localhost:8000/summarize` | Summarize the paper |
| `POST` | `http://localhost:8000/keypoints` | Extract key points |
| `GET`  | `http://localhost:8000/docs` | Swagger UI |

> ⚠️ **Keep this notebook running** while using the MERN app — the last cell starts the server.

---
## 📦 STEP 1 — Install All Required Libraries

Same libraries as before. Run once — subsequent runs are instant from cache.

> 💡 **VS Code:** Press `Ctrl+Shift+P` → "Python: Select Interpreter" to pick your environment first.

In [1]:
import sys

print('📦 Installing required libraries...')
print('   This may take a few minutes on first run.\n')

!{sys.executable} -m pip install --quiet \
    PyMuPDF \
    nltk \
    sentence-transformers \
    chromadb \
    transformers \
    torch \
    fastapi \
    uvicorn \
    python-multipart \
    nest-asyncio

print('✅ All libraries installed!')

📦 Installing required libraries...
   This may take a few minutes on first run.

✅ All libraries installed!


---
## 📥 STEP 2 — Download NLTK Data

In [2]:
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('✅ NLTK data ready!')

✅ NLTK data ready!


---
## 🤖 STEP 3 — Load AI Models

Same two models as before — no extra download.

| Model | Role | Size |
|---|---|---|
| `all-MiniLM-L6-v2` | Converts text → vectors for similarity search | ~90 MB |
| `google/flan-t5-base` | Reads context + question → generates answer | ~990 MB |

> Both are cached after first download — restart loads them in ~30 seconds.

In [3]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Running on: {device.upper()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

print('\n📥 Loading embedding model...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedding model ready!')

print('\n📥 Loading Flan-T5 model (google/flan-t5-base)...')
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
gen_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
gen_model = gen_model.to(device)
gen_model.eval()
print('✅ Flan-T5 ready!')

print('\n🎉 All models loaded!')

C:\Users\lahir\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️  Running on: CPU

📥 Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6383.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!

📥 Loading Flan-T5 model (google/flan-t5-base)...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3683.35it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Flan-T5 ready!

🎉 All models loaded!


---
## 🗄️ STEP 4 — Initialize ChromaDB

We use **two separate ChromaDB collections** this time:

| Collection | Purpose |
|---|---|
| `paper_chunks` | Stores all text chunks for retrieval during Q&A |
| `paper_sections` | Stores larger section-level text for summarization and context building |

Using two collections means:
- **Q&A** uses fine-grained chunks → precise answers
- **Summarize/Keypoints** uses broader sections → better overview

In [4]:
import chromadb
import os

NOTEBOOK_DIR = os.getcwd()
CHROMA_PATH  = os.path.join(NOTEBOOK_DIR, 'chroma_db')

print(f'📁 ChromaDB path: {CHROMA_PATH}')
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

def get_or_create_collection(name):
    try:
        col = chroma_client.get_collection(name=name)
        print(f'✅ Loaded existing collection "{name}" ({col.count()} items)')
        return col
    except Exception:
        col = chroma_client.create_collection(
            name=name,
            metadata={'hnsw:space': 'cosine'}
        )
        print(f'✅ Created new collection "{name}"')
        return col

chunks_collection   = get_or_create_collection('paper_chunks')
sections_collection = get_or_create_collection('paper_sections')

# Global variable to store the document-level summary (built at upload time)
document_overview = ''

print('\n✅ ChromaDB ready!')

📁 ChromaDB path: f:\CAMPUS\Academic Semester 7\Advanced AI\Project\ai-research-ppr-assistant-v2\chroma_db
✅ Created new collection "paper_chunks"
✅ Created new collection "paper_sections"

✅ ChromaDB ready!


---
## 🛠️ STEP 5 — Enhanced Pipeline Functions

This is the core of the accuracy improvements. Each function is explained below.

### Key improvements:

**1. Smarter Chunking**
- Chunk size increased from 500 → **1000 characters** so each chunk carries more context
- Overlap increased from 100 → **200 characters** so no context is lost at boundaries
- Section-level chunks (3000 chars) created separately for broad analysis

**2. Document Overview Generation**
- At upload time, we read the **first + last 20% of the paper** to build a document overview
- This overview is injected into every single prompt so the model always knows what paper it is reading
- Like giving someone the executive summary before asking them detailed questions

**3. Two-Stage Retrieval**
- **Stage 1:** Vector similarity search — retrieves top 8 chunks most similar to the question
- **Stage 2:** Keyword re-ranking — re-scores those 8 by counting keyword matches with the question
- Final top 6 chunks are selected — more context, better answers

**4. Structured Prompts**
- Each task (Q&A, summarize, keypoints) has its own carefully worded prompt
- Prompts instruct the model to be specific, cite evidence, and avoid vague answers

**5. Answer Post-Processing**
- Cleans up repetitive or incomplete model output
- Ensures the answer is never just an echo of the question

In [7]:
import re
import fitz
from nltk.tokenize import sent_tokenize
from collections import Counter


# ─────────────────────────────────────────────────────────────
# 1. PDF TEXT EXTRACTION
# ─────────────────────────────────────────────────────────────
def extract_text(pdf_bytes: bytes) -> tuple:
    """
    Extracts text from a PDF and returns:
    - full_text: entire paper as one string
    - page_texts: list of text per page (used for section building)
    """
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')
    page_texts = []
    for page in doc:
        page_texts.append(page.get_text())
    doc.close()
    full_text = '\n'.join(page_texts)
    return full_text, page_texts


# ─────────────────────────────────────────────────────────────
# 2. TEXT CLEANING
# ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Removes noise from PDF text while preserving meaningful structure.
    More careful than before — avoids removing numbers or citations.
    """
    text = text.replace('\x0c', ' ')             # Page breaks
    text = text.replace('\t', ' ')               # Tabs
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Non-ASCII
    text = re.sub(r'http\S+|www\.\S+', '', text) # URLs
    text = re.sub(r'\S+@\S+', '', text)          # Emails
    # Remove lines that are just page numbers or single characters
    lines = text.split('\n')
    lines = [l for l in lines if len(l.strip()) > 3]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)      # Max 2 consecutive newlines
    text = re.sub(r' {2,}', ' ', text)           # Multiple spaces
    return text.strip()


# ─────────────────────────────────────────────────────────────
# 3. ENHANCED CHUNKING — respects sentence boundaries
# ─────────────────────────────────────────────────────────────
def create_chunks(sentences: list,
                  chunk_size: int = 1000,
                  overlap: int = 200) -> list:
    """
    Creates overlapping text chunks that always end on a sentence boundary.

    Improvements over basic version:
    - chunk_size 1000 (was 500) — more context per chunk
    - overlap 200 (was 100) — less context lost at boundaries
    - Never cuts a sentence in half

    Args:
        sentences:  List of sentences from sent_tokenize
        chunk_size: Target max characters per chunk
        overlap:    Characters of overlap between consecutive chunks

    Returns:
        List of text chunk strings
    """
    chunks  = []
    current = []
    current_len = 0

    for sentence in sentences:
        s_len = len(sentence)
        if current_len + s_len <= chunk_size:
            current.append(sentence)
            current_len += s_len
        else:
            if current:
                chunks.append(' '.join(current))
            # Build overlap: keep sentences from end of current chunk
            overlap_sents = []
            overlap_len   = 0
            for s in reversed(current):
                if overlap_len + len(s) <= overlap:
                    overlap_sents.insert(0, s)
                    overlap_len += len(s)
                else:
                    break
            current     = overlap_sents + [sentence]
            current_len = sum(len(s) for s in current)

    if current:
        chunks.append(' '.join(current))

    return chunks


def create_sections(page_texts: list, pages_per_section: int = 3) -> list:
    """
    Groups pages into larger sections for high-level analysis.
    Each section covers ~3 pages, giving the model broader context
    for summarization and key point extraction.

    Args:
        page_texts:        List of text strings per page
        pages_per_section: How many pages per section

    Returns:
        List of section strings
    """
    sections = []
    for i in range(0, len(page_texts), pages_per_section):
        section = ' '.join(page_texts[i:i + pages_per_section])
        section = clean_text(section)
        if len(section.strip()) > 100:
            sections.append(section)
    return sections


# ─────────────────────────────────────────────────────────────
# 4. DOCUMENT OVERVIEW BUILDER
#    Reads beginning + end of paper to build a global summary
#    that is injected into every prompt.
# ─────────────────────────────────────────────────────────────
def build_document_overview(full_text: str) -> str:
    """
    Builds a compact document overview by combining:
    - First 20% of the paper (usually abstract + introduction)
    - Last 15% of the paper (usually conclusion + references section start)

    This overview is stored globally and prepended to every prompt,
    so the model always knows the paper's topic and scope before
    it reads the retrieved chunks.

    Args:
        full_text: Complete cleaned text of the paper

    Returns:
        Compact overview string (max 1200 characters)
    """
    total = len(full_text)
    intro_end       = int(total * 0.20)
    conclusion_start= int(total * 0.85)

    intro_text      = full_text[:intro_end]
    conclusion_text = full_text[conclusion_start:]

    # Take the first 700 chars of intro and first 500 chars of conclusion
    overview = (
        '[PAPER BEGINNING]\n' + intro_text[:700].strip() +
        '\n\n[PAPER CONCLUSION]\n' + conclusion_text[:500].strip()
    )
    return overview


# ─────────────────────────────────────────────────────────────
# 5. STORE CHUNKS + SECTIONS IN CHROMADB
# ─────────────────────────────────────────────────────────────
def store_all(chunks: list, sections: list):
    """
    Stores both fine-grained chunks and broad sections in their
    respective ChromaDB collections.

    - chunks_collection   → used for Q&A retrieval
    - sections_collection → used for summarization and key points
    """
    global chunks_collection, sections_collection

    # Clear old data
    for name in ['paper_chunks', 'paper_sections']:
        try:
            chroma_client.delete_collection(name)
        except Exception:
            pass

    chunks_collection = chroma_client.create_collection(
        name='paper_chunks', metadata={'hnsw:space': 'cosine'}
    )
    sections_collection = chroma_client.create_collection(
        name='paper_sections', metadata={'hnsw:space': 'cosine'}
    )

    # Embed and store chunks
    chunk_embeddings = embedding_model.encode(
        chunks, batch_size=32, convert_to_numpy=True, show_progress_bar=False
    )
    chunks_collection.add(
        ids        = [str(i) for i in range(len(chunks))],
        documents  = chunks,
        embeddings = chunk_embeddings.tolist()
    )

    # Embed and store sections
    section_embeddings = embedding_model.encode(
        sections, batch_size=16, convert_to_numpy=True, show_progress_bar=False
    )
    sections_collection.add(
        ids        = [str(i) for i in range(len(sections))],
        documents  = sections,
        embeddings = section_embeddings.tolist()
    )


# ─────────────────────────────────────────────────────────────
# 6. TWO-STAGE RETRIEVAL
#    Stage 1: Vector similarity (cast wide net)
#    Stage 2: Keyword re-ranking (pick the most on-topic)
# ─────────────────────────────────────────────────────────────
def keyword_score(text: str, query: str) -> float:
    """
    Counts how many words from the query appear in the text.
    Used to re-rank chunks after vector retrieval.

    Args:
        text:  A retrieved chunk
        query: The user's question

    Returns:
        A score between 0.0 and 1.0
    """
    # Remove common stopwords before scoring
    from nltk.corpus import stopwords
    stop = set(stopwords.words('english'))

    query_words = set(w.lower() for w in query.split() if w.lower() not in stop)
    text_words  = set(w.lower() for w in text.split())

    if not query_words:
        return 0.0

    matches = query_words.intersection(text_words)
    return len(matches) / len(query_words)


def retrieve_context(query: str,
                     collection,
                     initial_k: int = 8,
                     final_k:   int = 6) -> str:
    """
    Two-stage retrieval for maximum relevance:

    Stage 1 — Vector search:
        Retrieve top `initial_k` (8) chunks by cosine similarity.
        Casts a wide net to avoid missing relevant content.

    Stage 2 — Keyword re-ranking:
        Score each of the 8 chunks by keyword overlap with the query.
        Combine vector score (70%) + keyword score (30%) for final ranking.
        Return top `final_k` (6) chunks.

    More chunks + re-ranking = much more relevant context for the model.

    Args:
        query:     The user's question or task description
        collection: ChromaDB collection to search
        initial_k: How many chunks to fetch in stage 1
        final_k:   How many to keep after re-ranking

    Returns:
        Combined context string from final_k best chunks
    """
    available = collection.count()
    if available == 0:
        return ''

    k1 = min(initial_k, available)
    k2 = min(final_k,   available)

    # Stage 1: vector search
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k1,
        include=['documents', 'distances']
    )

    docs      = results['documents'][0]
    distances = results['distances'][0]   # Cosine distance (lower = more similar)

    # Convert distance → similarity score (0 to 1)
    vector_scores = [1.0 - d for d in distances]

    # Stage 2: keyword re-ranking
    scored = []
    for doc, vs in zip(docs, vector_scores):
        ks      = keyword_score(doc, query)
        combined = 0.70 * vs + 0.30 * ks   # Weighted combination
        scored.append((combined, doc))

    # Sort by combined score descending
    scored.sort(key=lambda x: x[0], reverse=True)

    # Return top final_k chunks joined together
    top_chunks = [doc for _, doc in scored[:k2]]
    return '\n\n'.join(top_chunks)


# ─────────────────────────────────────────────────────────────
# 7. STRUCTURED PROMPT BUILDERS
#    Each task gets its own carefully crafted prompt.
#    The document_overview is always included so the model
#    knows what paper it is reading.
# ─────────────────────────────────────────────────────────────
def build_qa_prompt(question: str, context: str) -> str:
    """
    Builds a question-answering prompt.

    Structure:
    1. Document overview (always present — gives global context)
    2. Retrieved relevant chunks (specific context for this question)
    3. Clear instruction to answer from the text
    4. The question itself

    This structure ensures the model always knows:
    - What paper it is reading (overview)
    - What section is relevant (retrieved context)
    - What exactly to answer (question)
    """
    return (
        f'You are an expert research paper analyst. '
        f'Answer the question using ONLY the information provided below. '
        f'Be specific, accurate, and concise. '
        f'If the answer is not in the text, say "This information is not found in the paper."\n\n'
        f'PAPER OVERVIEW:\n{document_overview[:800]}\n\n'
        f'RELEVANT SECTIONS FROM THE PAPER:\n{context[:1800]}\n\n'
        f'QUESTION: {question}\n\n'
        f'ANSWER:'
    )


def build_summary_prompt(context: str) -> str:
    """
    Builds a summarization prompt.
    Uses section-level context (broader than Q&A chunks).
    Instructs the model to cover all key aspects of the paper.
    """
    return (
        f'You are an expert academic summarizer. '
        f'Write a clear, structured summary of this research paper.\n'
        f'Your summary MUST cover: (1) the problem being solved, '
        f'(2) the proposed approach or methodology, '
        f'(3) key results or findings, '
        f'(4) the main conclusion.\n'
        f'Write 4 to 6 sentences. Be specific — do not write vague generalities.\n\n'
        f'PAPER OVERVIEW:\n{document_overview[:800]}\n\n'
        f'PAPER CONTENT:\n{context[:2000]}\n\n'
        f'SUMMARY:'
    )


def build_keypoints_prompt(context: str) -> str:
    """
    Builds a key points extraction prompt.
    Instructs the model to identify concrete, specific contributions.
    """
    return (
        f'You are an expert research analyst. '
        f'Extract the most important contributions and findings from this research paper.\n'
        f'List exactly 4 key points. Each point must:\n'
        f'- Be a specific, concrete finding or contribution\n'
        f'- Be stated in one or two clear sentences\n'
        f'- Be directly supported by the paper content below\n\n'
        f'PAPER OVERVIEW:\n{document_overview[:800]}\n\n'
        f'PAPER CONTENT:\n{context[:2000]}\n\n'
        f'KEY POINTS:'
    )


# ─────────────────────────────────────────────────────────────
# 8. ANSWER GENERATION WITH POST-PROCESSING
# ─────────────────────────────────────────────────────────────
def generate_response(prompt: str, max_tokens: int = 300) -> str:
    """
    Generates a response using Flan-T5 with improved decoding settings.

    Improvements over basic version:
    - max_tokens increased to 300 (was 256) for more complete answers
    - length_penalty=1.5 encourages longer, more complete answers
    - repetition_penalty=2.0 strongly discourages repeated phrases
    - no_repeat_ngram_size=4 (was 3) prevents longer repeated sequences

    Args:
        prompt:     Complete prompt with overview + context + question
        max_tokens: Maximum generated answer length

    Returns:
        Cleaned, post-processed answer string
    """
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=1024,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens       = max_tokens,
            num_beams            = 5,        # More beams = better quality (was 4)
            early_stopping       = True,
            no_repeat_ngram_size = 4,        # Prevent repeated 4-grams (was 3)
            length_penalty       = 1.5,      # Reward longer, complete answers
            repetition_penalty   = 2.0       # Strongly penalise repetition
        )

    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return post_process(raw)


def post_process(text: str) -> str:
    """
    Cleans up the model's raw output.

    - Removes accidental prompt leakage (e.g. model repeating the word ANSWER:)
    - Removes duplicate consecutive sentences
    - Strips leading/trailing whitespace
    - Ensures output is never empty

    Args:
        text: Raw decoded model output

    Returns:
        Cleaned answer string
    """
    # Remove prompt artifacts
    for artifact in ['ANSWER:', 'Answer:', 'SUMMARY:', 'KEY POINTS:', 'KEY POINT']:
        text = text.replace(artifact, '').strip()

    # Remove duplicate consecutive sentences
    sentences = sent_tokenize(text)
    seen = []
    for s in sentences:
        s_clean = s.strip().lower()
        if s_clean not in [x.strip().lower() for x in seen]:
            seen.append(s)
    text = ' '.join(seen)

    # Final cleanup
    text = re.sub(r' {2,}', ' ', text).strip()

    if not text:
        return 'The model could not generate a response for this query. Please try rephrasing your question.'

    return text


print('✅ All enhanced pipeline functions defined!')

✅ All enhanced pipeline functions defined!


---
## 🌐 STEP 6 — FastAPI Application & Endpoints

Same 5 endpoints as before — fully compatible with your existing MERN app.
No changes needed on the MERN side.

### What's improved inside each endpoint:

**`/upload`**
- Now builds and stores the **document overview** at upload time
- Creates both `paper_chunks` and `paper_sections` collections
- Returns richer stats (pages, sections, chunks)

**`/chat`**
- Uses **two-stage retrieval** (8 → 6 chunks after re-ranking)
- Uses the **structured Q&A prompt** with document overview

**`/summarize`**
- Uses **section-level** ChromaDB collection (broader context)
- Uses **structured summary prompt** covering all aspects

**`/keypoints`**
- Uses **section-level** collection
- Instructs model to produce 4 **specific, concrete** points

In [8]:
import traceback
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel


app = FastAPI(
    title       = 'Research Paper AI API — Enhanced',
    description = 'High-accuracy RAG pipeline for research paper Q&A. Connect your MERN app to this.',
    version     = '2.0.0'
)

app.add_middleware(
    CORSMiddleware,
    allow_origins = [
        'http://localhost:3000',
        'http://localhost:5000',
        'http://127.0.0.1:3000',
        'http://127.0.0.1:5000',
    ],
    allow_methods = ['*'],
    allow_headers = ['*'],
)


class ChatRequest(BaseModel):
    question: str


# ── GET /health ───────────────────────────────────────────────
@app.get('/health')
def health_check():
    return {
        'status'         : 'ok',
        'message'        : 'Research Paper AI API (Enhanced) running on port 8000',
        'chunks_loaded'  : chunks_collection.count(),
        'sections_loaded': sections_collection.count(),
        'paper_ready'    : chunks_collection.count() > 0,
        'overview_ready' : len(document_overview) > 0
    }


# ── POST /upload ──────────────────────────────────────────────
@app.post('/upload')
async def upload_paper(file: UploadFile = File(...)):
    """
    Full pipeline on upload:
    1. Extract text from PDF (page by page)
    2. Clean the text
    3. Build document overview (intro + conclusion)
    4. Create fine-grained chunks (1000 chars, 200 overlap)
    5. Create section-level groups (3 pages each)
    6. Embed and store both in ChromaDB

    MERN usage:
        const formData = new FormData();
        formData.append('file', pdfFile);
        await axios.post('http://localhost:8000/upload', formData);
    """
    global document_overview

    if not file.filename.lower().endswith('.pdf'):
        raise HTTPException(status_code=400, detail='Only PDF files are supported.')

    try:
        pdf_bytes = await file.read()

        # Step 1: Extract
        raw_text, page_texts = extract_text(pdf_bytes)
        if not raw_text.strip():
            raise HTTPException(
                status_code=422,
                detail='No text found. This may be a scanned or image-only PDF.'
            )

        # Step 2: Clean
        cleaned_text = clean_text(raw_text)
        cleaned_pages = [clean_text(p) for p in page_texts]

        # Step 3: Build document overview (stored globally)
        document_overview = build_document_overview(cleaned_text)

        # Step 4: Fine-grained chunks for Q&A
        sentences = sent_tokenize(cleaned_text)
        chunks    = create_chunks(sentences, chunk_size=1000, overlap=200)

        # Step 5: Section-level groups for summarization
        sections = create_sections(cleaned_pages, pages_per_section=3)

        # Step 6: Embed + store both
        store_all(chunks, sections)

        return {
            'status'          : 'success',
            'filename'        : file.filename,
            'total_pages'     : len(page_texts),
            'total_chunks'    : len(chunks),
            'total_sections'  : len(sections),
            'total_characters': len(cleaned_text),
            'overview_built'  : True,
            'message'         : 'Paper fully processed — ready for accurate Q&A!'
        }

    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc() # Print full error to notebook console
        raise HTTPException(status_code=500, detail=f'Processing error: {str(e)}')


# ── POST /chat ────────────────────────────────────────────────
@app.post('/chat')
def chat(request: ChatRequest):
    """
    Ask a question about the paper.

    Pipeline:
    1. Two-stage retrieval: vector search (top 8) → keyword re-rank (top 6)
    2. Build structured Q&A prompt with document overview + retrieved context
    3. Generate answer with Flan-T5
    4. Post-process and return

    MERN usage:
        await axios.post('http://localhost:8000/chat', { question: userMessage });

    Request:  { "question": "What dataset was used?" }
    Response: { "question": "...", "answer": "..." }
    """
    if chunks_collection.count() == 0:
        raise HTTPException(status_code=400, detail='No paper uploaded yet. Please use /upload first.')
    if not request.question.strip():
        raise HTTPException(status_code=400, detail='Question cannot be empty.')

    try:
        # Two-stage retrieval from fine-grained chunks
        context = retrieve_context(
            request.question,
            chunks_collection,
            initial_k=8,
            final_k=6
        )

        # Structured Q&A prompt
        prompt = build_qa_prompt(request.question, context)
        answer = generate_response(prompt, max_tokens=300)

        return {'question': request.question, 'answer': answer}

    except Exception as e:
        traceback.print_exc() # Print full error to notebook console
        raise HTTPException(status_code=500, detail=str(e))


# ── POST /summarize ───────────────────────────────────────────
@app.post('/summarize')
def summarize():
    """
    Generate a structured summary covering problem, approach, results, and conclusion.

    Uses section-level ChromaDB collection for broader context.

    MERN usage:
        await axios.post('http://localhost:8000/summarize');

    Response: { "summary": "..." }
    """
    if sections_collection.count() == 0:
        raise HTTPException(status_code=400, detail='No paper uploaded yet.')

    try:
        # Use section-level collection for broader context
        context = retrieve_context(
            'problem statement methodology approach results findings conclusion contribution',
            sections_collection,
            initial_k=6,
            final_k=4
        )

        prompt  = build_summary_prompt(context)
        summary = generate_response(prompt, max_tokens=350)

        return {'summary': summary}

    except Exception as e:
        traceback.print_exc() # Print full error to notebook console
        raise HTTPException(status_code=500, detail=str(e))


# ── POST /keypoints ───────────────────────────────────────────
@app.post('/keypoints')
def key_points():
    """
    Extract 4 specific, concrete key contributions from the paper.

    Uses section-level collection for broader context.

    MERN usage:
        await axios.post('http://localhost:8000/keypoints');

    Response: { "key_points": "..." }
    """
    if sections_collection.count() == 0:
        raise HTTPException(status_code=400, detail='No paper uploaded yet.')

    try:
        context = retrieve_context(
            'key contributions novel findings experimental results performance improvement',
            sections_collection,
            initial_k=6,
            final_k=4
        )

        prompt = build_keypoints_prompt(context)
        points = generate_response(prompt, max_tokens=300)

        return {'key_points': points}

    except Exception as e:
        traceback.print_exc() # Print full error to notebook console
        raise HTTPException(status_code=500, detail=str(e))


print('✅ FastAPI app defined — all 5 endpoints ready!')
print('\n📋 Endpoints:')
print('   GET  http://localhost:8000/health')
print('   POST http://localhost:8000/upload')
print('   POST http://localhost:8000/chat')
print('   POST http://localhost:8000/summarize')
print('   POST http://localhost:8000/keypoints')
print('   GET  http://localhost:8000/docs   ← Swagger UI')

✅ FastAPI app defined — all 5 endpoints ready!

📋 Endpoints:
   GET  http://localhost:8000/health
   POST http://localhost:8000/upload
   POST http://localhost:8000/chat
   POST http://localhost:8000/summarize
   POST http://localhost:8000/keypoints
   GET  http://localhost:8000/docs   ← Swagger UI


---
## 🚀 STEP 7 — Start the Server

Run this cell to start the API. **Keep it running** while using the MERN app.

### After starting:
- Visit **`http://localhost:8000/docs`** to test all endpoints in the browser
- Your MERN app connects via `PYTHON_API_URL=http://localhost:8000` in `.env`
- Upload a PDF via `/upload` before asking questions

### Tips for best answers:
| Ask this type of question | Example |
|---|---|
| Specific methodology | *"What algorithm or model is proposed?"* |
| Dataset details | *"What dataset was used for evaluation?"* |
| Results & metrics | *"What accuracy or performance was achieved?"* |
| Comparisons | *"How does this compare to baseline methods?"* |
| Limitations | *"What are the limitations mentioned?"* |

> ❌ Avoid very vague questions like *"Tell me about this paper"* — use `/summarize` for that instead.

In [ ]:
import nest_asyncio
import uvicorn
import asyncio

nest_asyncio.apply()

print('🚀 Starting Enhanced Research Paper AI API...')
print('=' * 55)
print('  Server URL  :  http://localhost:8000')
print('  Swagger Docs:  http://localhost:8000/docs')
print('  Health Check:  http://localhost:8000/health')
print('=' * 55)
print('⚠️  Keep this cell running while using the MERN app.')
print('   Press the Stop (■) button to shut down.\n')

# Fix for running uvicorn in a notebook environment
config = uvicorn.Config(app, host='0.0.0.0', port=8000, reload=False)
server = uvicorn.Server(config)
await server.serve()

🚀 Starting Enhanced Research Paper AI API...
  Server URL  :  http://localhost:8000
  Swagger Docs:  http://localhost:8000/docs
  Health Check:  http://localhost:8000/health
⚠️  Keep this cell running while using the MERN app.
   Press the Stop (■) button to shut down.



INFO:     Started server process [1864]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:12071 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:12071 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:2831 - "POST /upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:11298 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:3410 - "POST /chat HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:11880 - "POST /summarize HTTP/1.1" 200 OK
INFO:     127.0.0.1:13309 - "POST /keypoints HTTP/1.1" 200 OK
INFO:     127.0.0.1:9115 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:5406 - "POST /chat HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:2575 - "POST /chat HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:1100 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:13233 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:11370 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:6881 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:6696 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:5425 - "POST /chat HTTP/1.1" 500 Internal Server Error
